# EXP-01 · Subset size vs finite-sampling effects

**질문:** 같은 설명을 평가할 때 subset 크기 $k$를 바꾸면 FC의 기준값이 바뀔까, 아니면 적은 draw 수 $B$ 때문에 추정값만 흔들릴까?

This notebook runs the exact same implementation as the command line. The current protocol comes from XAI Framework Atlas, revision 4. Start with `smoke`; use `development` for the planned development grid. Confirmation remains a separate, frozen protocol.

**CPU only.** Inputs are generated from a declared normal distribution and the model is an explicit scalar formula. No training, labels, external dataset or GPU is needed. See [experiment design](../docs/EXP-01.md) for the complete rationale and failure interpretation.


## 1. Open the repository and install the small core environment

**Colab:** [open this notebook](https://colab.research.google.com/github/theguardianofthebed/xai-metric-behavior/blob/exp01-atlas-protocol/notebooks/EXP-01.ipynb) and run the setup cell in a fresh Python runtime. It clones the `exp01-atlas-protocol` branch when no suitable checkout exists. **Local Jupyter:** start inside the repository, or set `REPO_DIR` to your checkout. After merging you may change `REF` to `main`.

The setup installs core dependencies only when their pinned versions are missing, then loads this checkout directly. It never updates an existing checkout silently.


In [ ]:
from pathlib import Path
import os, sys, subprocess, importlib.metadata as metadata

REPOSITORY = "https://github.com/theguardianofthebed/xai-metric-behavior.git"
REF = "exp01-atlas-protocol"
REPO_DIR = None  # Optionally set an existing local checkout path.

if REPO_DIR:
    repo = Path(REPO_DIR).expanduser().resolve()
else:
    repo = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "configs/exp01_smoke.yaml").exists()), None)
    if repo is None:
        repo = (Path("/content") if Path("/content").exists() else Path.cwd()) / "xai-metric-behavior"
        if not repo.exists():
            subprocess.run(["git", "clone", "--branch", REF, REPOSITORY, str(repo)], check=True)
if not (repo / "configs/exp01_smoke.yaml").exists():
    raise RuntimeError("This checkout does not contain EXP-01. Use the PR branch, or main after merge.")
os.chdir(repo)
pins = dict(line.strip().split("==", 1) for line in Path("requirements.txt").read_text().splitlines()
            if line.strip() and not line.startswith("#"))
missing = []
for package, version in pins.items():
    try:
        if metadata.version(package) != version:
            missing.append(package)
    except metadata.PackageNotFoundError:
        missing.append(package)
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
# Explicit source path also works in a kernel already started before editable installation.
sys.path.insert(0, str(repo / "src"))
print("Repository:", repo)
print("Python:", sys.version.split()[0])
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


## 2. Read the contract before running

- $g(x)=2.5x_1-2x_2+1.5x_3+x_4-0.5x_5$; input dimension $d=20$.
- Baseline is literal numeric zero. Preserve attribution signs; no absolute values, normalization or probability transform.
- Exact anchor $a=v$; candidate A moves feature 1's contribution to unused feature 6; B additionally moves feature 2 to unused feature 7. Candidate values and sparsity are preserved.
- $k$ is features selected **per draw**. $B$ is draws **per score**. Inputs and repeated evaluators are separate levels.
- One uniform feature permutation per draw supplies prefixes for different $k$. Budget prefixes and explanation variants share the same recorded plan.

Population prediction: $FC_{pop}(a;k)=Corr_j(a_j,v_j)$ for $1\leq k<20$, when both centered variances are positive. This is a conditional mathematical prediction; finite estimates may differ or be undefined.


In [ ]:
import yaml
PROFILE = "smoke"  # Change to "development" after reviewing the smoke outputs.
if PROFILE not in {"smoke", "development"}:
    raise ValueError("Use the separate freeze/confirmation step for reserved conditions.")
config_path = repo / "configs" / f"exp01_{PROFILE}.yaml"
config = yaml.safe_load(config_path.read_text())
print(yaml.safe_dump(config, sort_keys=False))


## 3. Run and retain every output

Each run gets a new timestamped directory. The evaluator saves inputs, population references, scores, undefined reasons, paired comparisons, and draw-level traces. Increasing $B$ is not assumed to improve every individual estimate.


In [ ]:
from datetime import datetime, timezone
from xai_metric_behavior.experiment import run_experiment
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
output_dir = repo / "results" / f"exp01-{PROFILE}-{stamp}"
run_experiment(config_path, output_dir)
print("Saved:", output_dir)


## 4. Inspect the evidence and failures together

`population_scores.csv` is the analytical reference for each input and explanation. `scores.csv` contains repeated finite estimates. `exact_enumeration.csv` independently checks declared cases by actual model deletion. `contrast_summary.csv` uses jointly valid paired cases; validity differences can change the conditional estimand.

Empty failure logs do not establish general metric reliability. Nonempty logs may contain expected zero-variance correlations, which must remain undefined rather than become low scores.


In [ ]:
import csv, json
from IPython.display import display, Markdown, Image

def preview_csv(name, rows=8):
    path = output_dir / name
    with path.open(newline="") as stream:
        reader = csv.DictReader(stream)
        fields = reader.fieldnames or []
        values = list(reader)
    print(f"{name}: {len(values)} rows")
    for row in values[:rows]:
        print(row)
    return values

manifest = json.loads((output_dir / "manifest.json").read_text())
print(json.dumps(manifest, indent=2)[:8000])
for name in ["population_scores.csv", "exact_enumeration.csv", "score_summary.csv",
             "contrast_summary.csv", "failures.csv"]:
    preview_csv(name)
for figure in sorted((output_dir / "figures").glob("*.png")):
    display(Markdown(f"### {figure.stem.replace('_', ' ')}"))
    display(Image(filename=str(figure)))


## 5. Follow one saved computation

$U_S$ adds the supplied attribution values at selected coordinates. $V_S=g(x)-g(x_{S\leftarrow0})$ measures the output change from replacing those same coordinates in the original input. Their correlation is calculated **within one input**, across draws.

The NPZ trace preserves the actual feature orderings, $U$, $V$ and perturbed outputs. A smaller budget is a prefix of the recorded draw stream; it is not a new independent experiment.


In [ ]:
import numpy as np
trace_path = sorted((output_dir / "traces").glob("*.npz"))[0]
with np.load(trace_path, allow_pickle=False) as trace:
    print("Trace:", trace_path.name)
    print({key: trace[key].shape for key in trace.files})
    for key in ["permutations", "U", "V", "perturbed_outputs"]:
        if key in trace:
            print(key, trace[key].reshape(-1)[:20])


## 6. Development first, then freeze confirmation

Run the notebook again with `PROFILE="development"` after understanding the smoke run. Inspect the population margins, failure rates, uncertainty and relevant paired contrasts. Provisional counts are computational choices, not a power calculation.

When ready to use reserved conditions, freeze the prediction **before** opening their outputs. Use the completed development output path printed above:

```bash
python -m xai_metric_behavior freeze --config configs/exp01_confirmation.yaml --development-run results/YOUR_DEVELOPMENT_RUN --output protocol-freeze.json
python -m xai_metric_behavior run --config configs/exp01_confirmation.yaml --freeze protocol-freeze.json --output results/exp01-confirmation
```

Reserved subset sizes are `{2,7,13}`. Fresh inputs and evaluator streams accompany them. The freeze guards matching code/configuration, not publication-grade statistical power; changing a scientific prediction after seeing confirmation requires fresh reserved conditions.

To retain a run outside this runtime, download the **complete output directory** and its freeze file, not only figures. This notebook does not upload research outputs automatically.


## Research Learning Summary

**필요성:** subset 크기의 효과와 sampling 오차를 구분해야 같은 설명의 점수가 달라진 이유를 해석할 수 있다.  
**직관적인 예시/이해:** 설명은 그대로 두고 시험 문제를 뽑는 방식과 문제 수만 바꾼다. 전체 문제를 모두 계산한 기준값과 적게 뽑은 점수를 비교한다.  
**공식적인 정의:** $U_S=\sum_{j\in S}a_j$, $V_S=g(x)-g(x_{S\leftarrow0})$, $\widehat{FC}_B=Corr_{r=1}^B(U_{S_r},V_{S_r})$. 균등한 고정 크기 subset과 additive target에서 $FC_{pop}=Corr_j(a_j,v_j)$.  
**근거/그게 되는 이유:** subset 합의 공분산과 두 분산에 공통으로 생기는 계수가 Pearson 비율에서 상쇄된다. 이 결론은 population 값에 관한 것이며, finite Pearson 추정량의 오차·퇴화 확률이 동일하다는 뜻은 아니다.
